In [0]:
# ── Configuration ──────────────────────────────────────────────
dbutils.widgets.text("CATALOG", "dev")

CATALOG = dbutils.widgets.get("CATALOG")
print(f"Catalog parameter : {CATALOG}")

In [0]:
%sql
-- ── Catalog ─────────────────────────────────────────────────
-- IDENTIFIER(:CATALOG) safely binds the widget value as a SQL identifier.
CREATE CATALOG IF NOT EXISTS IDENTIFIER(:CATALOG)
COMMENT 'Global wholesale distribution business. The business  sells products to customers across multiple countries and regions and works with a network of suppliers.';

In [0]:
# ── Schemas (medallion + ops) ──────────────────────────────────
SCHEMAS = {
    "bronze":  "Ingested source data preserving native source structure",
    "silver":  "Cleaned, standardized, and validated data",
    "gold":    "Business-ready tables, aggregates, and analytics",
}

for schema, comment in SCHEMAS.items():
    spark.sql(
        "CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:name) COMMENT :comment",
        args={"name": f"{CATALOG}.{schema}", "comment": comment},
    )
    print(f"✓ Schema: {CATALOG}.{schema}")

In [0]:
%sql
SHOW SCHEMAS IN dev;

In [0]:
# 6. Verify source read access
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .table("samples.tpch.customer")
)

print("Source read access verified successfully.")
print("Columns:", df.columns)
print("Sample records:")
display(df.limit(5))

In [0]:
# Verify write access by creating a temporary test table
TEST_TABLE = f"{CATALOG}.bronze._write_access_test"

spark.sql(f"""
    CREATE TABLE {TEST_TABLE} (
        test_id INT
    ) USING DELTA
""")

In [0]:
spark.sql(f"DROP TABLE {TEST_TABLE}")

print("Write access verified successfully.")